## Aramina P. Galario

DS3A

In [3]:
import numpy as np
from sklearn.datasets import load_iris
from collections import Counter

iris = load_iris()
X = iris.data
y = iris.target

**This uses entropy and information gain to decide splits.
At each node, it selects the feature whose median-based split maximizes information gain.
The tree grows recursively until a maximum depth or pure leaf is reached.**

In [5]:
class DecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._grow_tree(X, y, 0)

    def entropy(self, y):
        counts = Counter(y)
        entropy = 0
        for cls in counts:
            p = counts[cls] / len(y)
            entropy -= p * np.log2(p)
        return entropy

    def best_features(self, X, y):
        best_feature = None
        best_gain = -1

        parent_entropy = self.entropy(y)

        for feature in range(X.shape[1]):
            threshold = np.median(X[:, feature])

            left = y[X[:, feature] <= threshold]
            right = y[X[:, feature] > threshold]

            if len(left) == 0 or len(right) == 0:
                continue

            child_entropy = (
                len(left)/len(y) * self.entropy(left) +
                len(right)/len(y) * self.entropy(right)
            )

            info_gain = parent_entropy - child_entropy

            if info_gain > best_gain:
                best_gain = info_gain
                best_feature = feature

        return best_feature

    def _grow_tree(self, X, y, depth):
        if len(set(y)) == 1 or depth == self.max_depth:
            return Counter(y).most_common(1)[0][0]

        feature = self.best_features(X, y)
        if feature is None:
            return Counter(y).most_common(1)[0][0]

        threshold = np.median(X[:, feature])
        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self._grow_tree(X[left_mask], y[left_mask], depth + 1),
            "right": self._grow_tree(X[right_mask], y[right_mask], depth + 1)
        }

    def predict(self, X):
        return np.array([self._predict(x, self.tree) for x in X])

    def _predict(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node["feature"]] <= node["threshold"]:
            return self._predict(x, node["left"])
        else:
            return self._predict(x, node["right"])


In [7]:
model = DecisionTree(max_depth=3)
model.fit(X, y)

predictions = model.predict(X)
print(predictions[:10])

accuracy = np.mean(predictions == y)
print("Accuracy:", accuracy)


[0 0 0 0 0 1 0 0 0 0]
Accuracy: 0.9133333333333333
